In [1]:
# PyTorch Experiment Tracking Demo (Colab Ready)

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import itertools
import json
import random
import numpy as np

In [2]:
# 1. Reproducibility

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed()

In [3]:
# 2. Generate Dataset

X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_classes=2)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

In [4]:
# 3. Model Builder

class SimpleNN(nn.Module):
    def __init__(self, input_dim, layers):
        super().__init__()
        modules = []
        prev_dim = input_dim

        for h in layers:
            modules.append(nn.Linear(prev_dim, h))
            modules.append(nn.ReLU())
            prev_dim = h

        modules.append(nn.Linear(prev_dim, 1))
        modules.append(nn.Sigmoid())

        self.model = nn.Sequential(*modules)

    def forward(self, x):
        return self.model(x)

In [5]:
# 4. Training Function

def train_model(config):
    model = SimpleNN(input_dim=20, layers=config["layers"])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])

    epochs = config["epochs"]

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = criterion(val_outputs, y_val).item()
        preds = (val_outputs > 0.5).float()
        accuracy = (preds == y_val).float().mean().item()

    return {
        "val_loss": val_loss,
        "val_accuracy": accuracy
    }

In [6]:
# 5. Experiment Logger

def save_result(result, config, filename="experiments.csv"):
    record = {**config, **result}
    record["config_json"] = json.dumps(config)

    df = pd.DataFrame([record])

    try:
        df_existing = pd.read_csv(filename)
        df = pd.concat([df_existing, df], ignore_index=True)
    except FileNotFoundError:
        pass

    df.to_csv(filename, index=False)

In [7]:
# 6. Run Experiments (Grid Search)

layer_options = [[16], [32], [64, 32]]
lr_options = [0.01, 0.001]
epoch_options = [10, 20]

configs = list(itertools.product(layer_options, lr_options, epoch_options))

for layers, lr, epochs in configs:
    config = {
        "layers": layers,
        "lr": lr,
        "epochs": epochs
    }

    result = train_model(config)
    save_result(result, config)

In [8]:
# 7. Analyze Results

df = pd.read_csv("experiments.csv")

print("\nTop Experiments by Accuracy:")
print(df.sort_values(by="val_accuracy", ascending=False).head())

print("\nTop Experiments by Loss:")
print(df.sort_values(by="val_loss").head())


Top Experiments by Accuracy:
      layers     lr  epochs  val_loss  val_accuracy  \
9   [64, 32]  0.010      20  0.185002         0.940   
5       [32]  0.010      20  0.281338         0.905   
8   [64, 32]  0.010      10  0.300066         0.890   
1       [16]  0.010      20  0.405931         0.865   
11  [64, 32]  0.001      20  0.570744         0.855   

                                        config_json  
9    {"layers": [64, 32], "lr": 0.01, "epochs": 20}  
5        {"layers": [32], "lr": 0.01, "epochs": 20}  
8    {"layers": [64, 32], "lr": 0.01, "epochs": 10}  
1        {"layers": [16], "lr": 0.01, "epochs": 20}  
11  {"layers": [64, 32], "lr": 0.001, "epochs": 20}  

Top Experiments by Loss:
     layers    lr  epochs  val_loss  val_accuracy  \
9  [64, 32]  0.01      20  0.185002         0.940   
5      [32]  0.01      20  0.281338         0.905   
8  [64, 32]  0.01      10  0.300066         0.890   
1      [16]  0.01      20  0.405931         0.865   
4      [32]  0.01      1

In [10]:
df

,layers,lr,epochs,val_loss,val_accuracy,config_json
0,[16],0.010,10,0.562269,0.780,"{""layers"": [16], ""lr"": 0.01, ""epochs"": 10}"
1,[16],0.010,20,0.405931,0.865,"{""layers"": [16], ""lr"": 0.01, ""epochs"": 20}"
2,[16],0.001,10,0.669081,0.665,"{""layers"": [16], ""lr"": 0.001, ""epochs"": 10}"
3,[16],0.001,20,0.665613,0.635,"{""layers"": [16], ""lr"": 0.001, ""epochs"": 20}"
4,[32],0.010,10,0.507890,0.845,"{""layers"": [32], ""lr"": 0.01, ""epochs"": 10}"
5,[32],0.010,20,0.281338,0.905,"{""layers"": [32], ""lr"": 0.01, ""epochs"": 20}"
6,[32],0.001,10,0.663154,0.645,"{""layers"": [32], ""lr"": 0.001, ""epochs"": 10}"
7,[32],0.001,20,0.642080,0.685,"{""layers"": [32], ""lr"": 0.001, ""epochs"": 20}"
8,"[64, 32]",0.010,10,0.300066,0.890,"{""layers"": [64, 32], ""lr"": 0.01, ""epochs"": 10}"
9,"[64, 32]",0.010,20,0.185002,0.940,"{""layers"": [64, 32], ""lr"": 0.01, ""epochs"": 20}"
